# Converting coral database to the LiPD format

## Overview

This notebook converts the coral database used in [Emile-Geay et al. (2015)](https://www.nature.com/articles/ngeo2608) from a Matlab structure to[LiPD](https://lipd.net) formatted datasets.   

## Prerequisites

| Concepts | Importance | Notes |
| --- | --- | --- |
| [Intro to Pandas](https://foundations.projectpythia.org/core/pandas.html) | Necessary | |
| [Understanding of LiPD format](https://lipd.net)| Helpful | Familiarity with the overall architecture|
| [Understanding of the LinkedEarth Ontology](http://linked.earth/ontology/)| Helpful | Familiarity with the classes and various properties|
| [Creating LiPD files with PyLiPD](http://linked.earth/pylipdTutorials/intro.html)|Necessary||

- **Time to learn**: 40 min.

***

## Imports

To figure out which library is more appropriate to load the `.mat` file, we first need to run the following cell: 

In [1]:
file_path = "../data/obs/coral_db_holo.mat"
with open(file_path, "rb") as f:
    print(f.read(4))

b'MATL'


Since, this is an older format, we need to use the `scipy.io.loadmat` function. 

In [2]:
#Read Matlab file
import scipy.io

#Manipulate data
import pandas as pd
import numpy as np
import json

#PyLiPD relevant functionalities
from pylipd.classes.dataset import Dataset
from pylipd.classes.archivetype import ArchiveType, ArchiveTypeConstants
from pylipd.classes.funding import Funding
from pylipd.classes.interpretation import Interpretation
from pylipd.classes.interpretationvariable import InterpretationVariable, InterpretationVariableConstants
from pylipd.classes.location import Location
from pylipd.classes.paleodata import PaleoData
from pylipd.classes.datatable import DataTable
from pylipd.classes.paleounit import PaleoUnit, PaleoUnitConstants
from pylipd.classes.paleovariable import PaleoVariable, PaleoVariableConstants
from pylipd.classes.person import Person
from pylipd.classes.publication import Publication
from pylipd.classes.resolution import Resolution
from pylipd.classes.variable import Variable
from pylipd.classes.model import Model
from pylipd.classes.chrondata import ChronData
from pylipd.classes.physicalsample import PhysicalSample
from pylipd.classes.compilation import Compilation

from pylipd import LiPD

***

## Exploring the coral database

In [3]:
data = scipy.io.loadmat(file_path)
coral_db = np.squeeze(data.get("C"))

Let's put this in a [Pandas `DataFrame`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) for further analysis: 

In [4]:
# Create a list to store flattened data
flattened_data = []

# Iterate over all elements in coral_db
for i, elem in enumerate(coral_db):
    if isinstance(elem, np.void) and elem.dtype.names is not None:
        # If it's a structured array (MATLAB struct), extract fields into a dictionary
        row_data = {field: elem[field] for field in elem.dtype.names}
    elif isinstance(elem, np.ndarray) and elem.size == 1:
        # If it's a single-value NumPy array, extract the scalar
        row_data = {f"Field_{i}": elem.item()}
    else:
        # Otherwise, keep the original value
        row_data = {f"Field_{i}": elem}

    flattened_data.append(row_data)

# Convert the list of dictionaries into a Pandas DataFrame
df_flattened = pd.DataFrame(flattened_data)

df_cleaned = df_flattened.map(lambda x: x.item() if isinstance(x, np.ndarray) and x.size == 1 else x)

Let's have a look at the first few rows:

In [5]:
df_cleaned.head()

,archive,lat,lon,site,reference,citekey,genus,species,sample_id,modern_id,...,enso_var_q,enso_var_err,seasonal_amp_q,seasonal_amp_err,enso_var_ratio,enso_var_ratio_q,enso_var_ratio_err,seasonal_amp_ratio,seasonal_amp_ratio_q,seasonal_amp_ratio_err
0,coral,-20.900000,165.482000,Bayes Islet,"Correge, pers. comm., 2013",this study,Porites,sp,Bayes1,BayesM,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,coral,-20.900000,165.482000,Bayes Islet,"Correge, pers. comm., 2013",this study,Porites,sp,BayesM,BayesM,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,coral,1.866667,202.583333,Christmas Is.,Woodroffe et al. [2003],Woodroffe_GRL03,Porites,sp.,XM1,Kiritimati_mod,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,coral,1.866667,202.583333,Christmas Is.,Woodroffe et al. [2003],Woodroffe_GRL03,Porites,sp.,XM9,Kiritimati_mod,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,coral,1.733333,202.800000,Christmas Is.,"McGregor et al., 2013",McGregor_ngeo2013,Porites,sp,XM35,Kiritimati_mod,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Let's have a look a unique entries for some of these columns.

In [6]:
print(df_cleaned['archive'].unique())
print(df_cleaned['reference'].unique())
print(df_cleaned['citekey'].unique())
print(df_cleaned['meas'].unique())

['coral']
['Correge, pers. comm., 2013' 'Woodroffe et al. [2003]'
 'McGregor et al., 2013' 'Cobb et al. [2013]' 'Tudhope et al. [2001]'
 'McGregor & Gagan [2004]' 'Woodroffe & Gagan [2000]'
 'Kilbourne et al. [2004]' 'Cobb et al. [2003]' 'Duprey et al. [2012]']
['this study' 'Woodroffe_GRL03' 'McGregor_ngeo2013' 'Cobb13' 'Tudhope2001'
 'McGregor_Gagan_GRL04' 'WoodroffeGagan_GRL00' 'Kilbourne2004' 'Cobb2003'
 'Duprey_Paleo_2012']
['\\delta^{18}O_{sw}' '\\delta^{18}O' 'SST']


Some observations:
1. The LiPD format takes full publication field that will need to be added.
2. All archivetypes are Coral, which is exepected
3. There are only three types of observation, which we can use to our advantage.

Let's transform these records into LiPD files!

## Converting the coral database to LiPD files

### Publications

Let's create 10 [`Publication` objects](https://pylipd.readthedocs.io/en/latest/api.html#pylipd.classes.publication.Publication) corresponding to the 10 distinct citations in the coral database and place them in a dictionary using the citekey as the key. 

Let's start with creating an empty dictionary that we will fill out as we go:

In [7]:
pub_dict ={}

#### McGregor and Gagan (2004)

McGregor, H. V. & Gagan, M. K. Western Pacific coral $\delta^{18}$O records of anomalous Holocene variability in the El Niño-Southern Oscillation. Geophys. Res. Lett. 31, L11204 (2004).

In [8]:
pub1 = Publication()
# first author
author1 = Person()
author1.setName("McGregor,H.V.")
# Second author
author2 = Person()
author2.setName("Gagan, M.K.")

pub1.setAuthors([author1,author2])
pub1.setTitle("Western Pacific coral $\delta^{18}$O  records of anomalous Holocene variability in the El Niño-Southern Oscillation.")
pub1.setJournal("Geophysical Research Letters")
pub1.setYear(2004)
pub1.setVolume("31")
pub1.setPages("L11204")

In [9]:
pub_dict['McGregor_Gagan_GRL04'] = pub1

#### McGregor et al. (2013)

Reference: McGregor, H. V. et al. A weak El Niño-Southern Oscillation with delayed seasonal growth around 4,300 years ago. Nature Geosci. 6, 949–953 (2013).

In [10]:
pub2 = Publication()
# first author
author1 = Person()
author1.setName("McGregor,H.V.")
# Second author
author2 = Person()
author2.setName("Fischer, M.J.")
#Third author
author3 = Person()
author3.setName("Gagan, M.K.")
#Fourth author
author4 = Person()
author4.setName("Fink, D.")
# Fifth author
author5 = Person()
author5.setName("Phipps, S.J.")
# Sixth author
author6 = Person()
author6.setName("Wong, H.")
#Seventh author
author7 = Person()
author7.setName("Woodroffe, C.D.")

pub2.setAuthors([author1,author2,author3,author4,author5,author6,author7])
pub2.setTitle("A weak El Niño-Southern Oscillation with delayed seasonal growth around 4,300 years ago.")
pub2.setJournal("Nature Geoscience")
pub2.setYear(2013)
pub2.setVolume("6")
pub2.setPages("949–953")

In [11]:
pub_dict['McGregor_ngeo2013'] = pub2

#### Cobb et al. (2013)

Reference: Cobb, K. M. et al. Highly variable El Niño-Southern Oscillation throughout the Holocene. Science 339, 67–70 (2013).

In [12]:
pub3 = Publication()
# first author
author1 = Person()
author1.setName("Cobb, K.M.")
# Second author
author2 = Person()
author2.setName("Westphal, N.")
#Third author
author3 = Person()
author3.setName("Sayani, H.R.")
#Fourth author
author4 = Person()
author4.setName("Watson, J.T.")
# Fifth author
author5 = Person()
author5.setName("Di Lorenzo, E.")
# Sixth author
author6 = Person()
author6.setName("Cheng, H.")
#Seventh author
author7 = Person()
author7.setName("Edwards, R.L.")
#Eigth author
author8 = Person()
author8.setName("Charles, C.D.")

pub3.setAuthors([author1,author2,author3,author4,author5,author6,author7,author8])
pub3.setTitle("Highly variable El Niño-Southern Oscillation throughout the Holocene.")
pub3.setJournal("Science")
pub3.setYear(2013)
pub3.setVolume("339")
pub3.setPages("67-70")
pub3.setDOI("10.1126/science.12282")

In [13]:
pub_dict['Cobb13'] = pub3

#### Cobb et al. (2003)

Reference: Cobb, K. M., Charles, C. D., Cheng, H. & Edwards, R. L. El Niño/Southern Oscillation and tropical Pacific climate during the last millennium. Nature 424, 271–276 (2003).

In [14]:
pub4 = Publication()
# first author
author1 = Person()
author1.setName("Cobb, K.M.")
#Second author
author2 = Person()
author2.setName("Charles, C.D.")
# Third author
author3 = Person()
author3.setName("Cheng, H.")
# Fourth author
author4 = Person()
author4.setName("Edwards, R.L.")

pub4.setAuthors([author1,author2,author3,author4])
pub4.setTitle("El Niño/Southern Oscillation and tropical Pacific climate during the last millennium.")
pub4.setJournal("Nature")
pub4.setYear(2003)
pub4.setVolume("424")
pub4.setPages("271-276")

In [15]:
pub_dict['Cobb2003']=pub4

#### Duprey et al. (2012)

Reference: Duprey, N. et al. Early mid-Holocene SST variability and surface-ocean water balance in the southwest Pacific. Paleoceanography 27, PA4207 (2012).

In [16]:
pub5 = Publication()
# first author
author1 = Person()
author1.setName("Duprey, N.")
# Second author
author2 = Person()
author2.setName("Lazareth, C.E.")
#Third author
author3 = Person()
author3.setName("Corrège, T.")
#Fourth author
author4 = Person()
author4.setName("Le Cornec, F.")
# Fifth author
author5 = Person()
author5.setName("Maes, C.")
# Sixth author
author6 = Person()
author6.setName("Pujol, N.")
#Seventh author
author7 = Person()
author7.setName("Madeng-Yogo, M.")
#Eigth author
author8 = Person()
author8.setName("Caquineau, S.")
#Ninth author
author9 = Person()
author9.setName("Soares Derome, C.")
#Tenth author
author10 = Person()
author10.setName("Cabioch, G.")

pub5.setAuthors([author1,author2,author3,author4,author5,author6,author7,author8,author9,author10])
pub5.setTitle("Early mid-Holocene SST variability and surface-ocean water balance in the southwest Pacific")
pub5.setJournal("Paleoceanography and Paleoclimatology")
pub5.setYear(2012)
pub5.setVolume("27")
pub5.setIssue("4")
pub5.setPages("PA4207")
pub5.setDOI("10.1029/2012PA002350")

In [17]:
pub_dict['Duprey_Paleo_2012']=pub5

#### Kilbourne et al. (2004)

Reference: Kilbourne, K. H., Quinn, T. M., Taylor, F. W., Delcroix, T. & Gouriou, Y. El Niño-Southern Oscillation-related salinity variations recorded in the skeletal geochemistry of a Porites coral from Espiritu Santo, Vanuatu. Paleoceanography 19, PA4002 (2004).

In [18]:
pub6 = Publication()
# first author
author1 = Person()
author1.setName("Kilbourne, K.H.")
# Second author
author2 = Person()
author2.setName("Quinn, T.M.")
#Third author
author3 = Person()
author3.setName("Taylor, F.W.")
#Fourth author
author4 = Person()
author4.setName("Delcroix, T.")
# Fifth author
author5 = Person()
author5.setName("Gouriou, Y.")

pub6.setAuthors([author1,author2,author3,author4,author5])
pub6.setTitle("El Niño-Southern Oscillation-related salinity variations recorded in the skeletal geochemistry of a Porites coral from Espiritu Santo, Vanuatu.")
pub6.setJournal("Paleoceanography and Paleoclimatology")
pub6.setYear(2004)
pub6.setVolume("19")
pub6.setPages("PA4002")

In [19]:
pub_dict['Kilbourne2004']=pub6

#### Woodroffe et al. (2000)

Reference: Woodroffe, C. D. & Gagan, M. K. Coral microatolls from the central Pacific record Late Holocene El Niño. Geophys. Res. Lett. 27, 1511–1514 (2000).

In [20]:
pub7 = Publication()
# first author
author1 = Person()
author1.setName("Woodroffe, C.D.")
# Second author
author2 = Person()
author2.setName("Gagan, M.K.")

pub7.setAuthors([author1,author2])
pub7.setTitle("Coral microatolls from the central Pacific record Late Holocene El Niño.")
pub7.setJournal("Geophysical Research Letters")
pub7.setYear(2000)
pub7.setVolume("27")
pub7.setPages("1511-1514")

In [21]:
pub_dict['WoodroffeGagan_GRL00'] = pub7

#### Woodroffe et al. (2003)

Reference: Woodroffe, C. D., Beech, M. R. & Gagan, M. K. Mid-late Holocene El Niño variability in the equatorial Pacific from coral microatolls. Geophys. Res. Lett. 30, 1358 (2003).

In [22]:
pub8 = Publication()
# first author
author1 = Person()
author1.setName("Woodroffe, C.D.")
# Second author
author2 = Person()
author2.setName("Beech, M.R.")
# Third author
author3 = Person()
author3.setName("Gagan, M.K.")

pub8.setAuthors([author1,author2,author3])
pub8.setTitle("Mid-late Holocene El Niño variability in the equatorial Pacific from coral microatolls.")
pub8.setJournal("Geophysical Research Letters")
pub8.setYear(2003)
pub8.setVolume("30")
pub8.setPages("1358")

In [23]:
pub_dict['Woodroffe_GRL03']=pub8

#### Tudhope et al. (2001)

Reference: Tudhope, A. W. et al. Variability in the El Niño-Southern Oscillation through a glacial-interglacial cycle. Science 291, 1511–1517 (2001).

In [24]:
pub9 = Publication()
# first author
author1 = Person()
author1.setName("Tudhope, A.W.")
# Second author
author2 = Person()
author2.setName("Chilcott, C.P.")
#Third author
author3 = Person()
author3.setName("McCulloch, M.T.")
#Fourth author
author4 = Person()
author4.setName("Cook, E.R.")
# Fifth author
author5 = Person()
author5.setName("Chappell, J.")
# Sixth author
author6 = Person()
author6.setName("Ellam, R.M.")
#Seventh author
author7 = Person()
author7.setName("Lea, D.W.")
#Eigth author
author8 = Person()
author8.setName("Lough, J.M.")
#Ninth author
author9 = Person()
author9.setName("Shimmield, G.B.")

pub9.setAuthors([author1,author2,author3,author4,author5,author6,author7,author8,author9])
pub9.setTitle("Variability in the El Niño-Southern Oscillation Through a Glacial-Interglacial Cycle")
pub9.setJournal("Science")
pub9.setYear(2001)
pub9.setVolume("291")
pub9.setIssue("5508")
pub9.setPages("1511-1517")
pub9.setDOI("10.1126/science.1057")

In [25]:
pub_dict['Tudhope2001']=pub9

#### Emile-Geay et al. (2015)

Records from personnal communication with Correge. 

this study

Reference: Emile-Geay, J., Cobb, K., Carré, M. et al. Links between tropical Pacific seasonal, interannual and orbital variability during the Holocene. Nature Geosci 9, 168–173 (2016). https://doi.org/10.1038/ngeo2608

In [26]:
pub10 = Publication()
# first author
author1 = Person()
author1.setName("Emile-Geay, J.")
# Second author
author2 = Person()
author2.setName("Cobb, K.M.")
#Third author
author3 = Person()
author3.setName("Carré, M.")
#Fourth author
author4 = Person()
author4.setName("Braconnot, P.")
# Fifth author
author5 = Person()
author5.setName("Leloup, J.")
# Sixth author
author6 = Person()
author6.setName("Zhou, Y.")
#Seventh author
author7 = Person()
author7.setName("Harrison, S.P.")
#Eigth author
author8 = Person()
author8.setName("Corrège, T.")
#Ninth author
author9 = Person()
author9.setName("McGregor, H.V.")
#Tenth author
author10 = Person()
author10.setName("Collins, M.")
#Eleventh author
author11 = Person()
author11.setName("Driscoll, R.")
#Twelveth author
author12 = Person()
author12.setName("Elliot, M.")
#Thirteenth author
author13 = Person()
author13.setName("Schneider, B.")
#Fourteenth author
author14 = Person()
author14.setName("Tudhope, A.")

pub10.setAuthors([author1,author2,author3,author4,author5,author6,author7,author8,author9,author10,author11,author12,author13,author14])
pub10.setTitle("Links between tropical Pacific seasonal, interannual and orbital variability during the Holocene")
pub10.setJournal("Nature Geoscience")
pub10.setYear(2016)
pub10.setVolume("9")
pub10.setPages("168-173")
pub10.setDOI("10.1038/ngeo2608")

In [27]:
pub_dict['this study']=pub10

For simplicity, let's store the publication objects in the Pandas Dataframe:

In [28]:
pub_column = []

for idx, row in df_cleaned.iterrows():
    pub_column.append(pub_dict[row['citekey']])

df_cleaned['Publication'] = pub_column
df_cleaned.head()

,archive,lat,lon,site,reference,citekey,genus,species,sample_id,modern_id,...,enso_var_err,seasonal_amp_q,seasonal_amp_err,enso_var_ratio,enso_var_ratio_q,enso_var_ratio_err,seasonal_amp_ratio,seasonal_amp_ratio_q,seasonal_amp_ratio_err,Publication
0,coral,-20.900000,165.482000,Bayes Islet,"Correge, pers. comm., 2013",this study,Porites,sp,Bayes1,BayesM,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<pylipd.classes.publication.Publication object...
1,coral,-20.900000,165.482000,Bayes Islet,"Correge, pers. comm., 2013",this study,Porites,sp,BayesM,BayesM,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<pylipd.classes.publication.Publication object...
2,coral,1.866667,202.583333,Christmas Is.,Woodroffe et al. [2003],Woodroffe_GRL03,Porites,sp.,XM1,Kiritimati_mod,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<pylipd.classes.publication.Publication object...
3,coral,1.866667,202.583333,Christmas Is.,Woodroffe et al. [2003],Woodroffe_GRL03,Porites,sp.,XM9,Kiritimati_mod,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<pylipd.classes.publication.Publication object...
4,coral,1.733333,202.800000,Christmas Is.,"McGregor et al., 2013",McGregor_ngeo2013,Porites,sp,XM35,Kiritimati_mod,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<pylipd.classes.publication.Publication object...


### Root metadata

LiPD files contain some root metadata, specifically:
* datasetname. This database uses the following convention: author.year.site_sampleID. We can create these automatically from the publication object and other information in the database.
* archiveType will be the same for every record in this case (i.e., coral) so let's not worry about it until we are ready to create a [`Dataset` object](https://pylipd.readthedocs.io/en/latest/api.html#pylipd.classes.dataset.Dataset).
* datasetId. This is a unique identifier for the dataset, we can create one automatically as we are ready to create the [`Dataset` object](https://pylipd.readthedocs.io/en/latest/api.html#pylipd.classes.dataset.Dataset).

Let's create the dataset names and store them in the DataFrame:

In [29]:
dataset_names = []

for idx, row in df_cleaned.iterrows():
    author = row['Publication'].getAuthors()[0].getName().split(',')[0] # get the last name of the first author
    year = row['Publication'].getYear()
    site=row['site'].split(",")[0].replace(" ","").replace(".","")
    sample=row['sample_id'].replace(" ","")

    dataset_names.append(author+"."+str(year)+"."+site+"_"+sample)    

In [30]:
df_cleaned['dataSetName'] = dataset_names

### Location

Let's create the [`Location` object](https://pylipd.readthedocs.io/en/latest/api.html#pylipd.classes.location.Location) from the information in the DataFrame and store it there: 

In [31]:
loc_info = []

for idx, row in df_cleaned.iterrows():
    loc_object = Location()
    loc_object.setLatitude(str(row['lat']))
    loc_object.setLongitude(str(row['lon']))
    loc_info.append(loc_object)

df_cleaned['Location']=loc_info
df_cleaned.head()    

,archive,lat,lon,site,reference,citekey,genus,species,sample_id,modern_id,...,seasonal_amp_err,enso_var_ratio,enso_var_ratio_q,enso_var_ratio_err,seasonal_amp_ratio,seasonal_amp_ratio_q,seasonal_amp_ratio_err,Publication,dataSetName,Location
0,coral,-20.900000,165.482000,Bayes Islet,"Correge, pers. comm., 2013",this study,Porites,sp,Bayes1,BayesM,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<pylipd.classes.publication.Publication object...,Emile-Geay.2016.BayesIslet_Bayes1,<pylipd.classes.location.Location object at 0x...
1,coral,-20.900000,165.482000,Bayes Islet,"Correge, pers. comm., 2013",this study,Porites,sp,BayesM,BayesM,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<pylipd.classes.publication.Publication object...,Emile-Geay.2016.BayesIslet_BayesM,<pylipd.classes.location.Location object at 0x...
2,coral,1.866667,202.583333,Christmas Is.,Woodroffe et al. [2003],Woodroffe_GRL03,Porites,sp.,XM1,Kiritimati_mod,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<pylipd.classes.publication.Publication object...,Woodroffe.2003.ChristmasIs_XM1,<pylipd.classes.location.Location object at 0x...
3,coral,1.866667,202.583333,Christmas Is.,Woodroffe et al. [2003],Woodroffe_GRL03,Porites,sp.,XM9,Kiritimati_mod,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<pylipd.classes.publication.Publication object...,Woodroffe.2003.ChristmasIs_XM9,<pylipd.classes.location.Location object at 0x...
4,coral,1.733333,202.800000,Christmas Is.,"McGregor et al., 2013",McGregor_ngeo2013,Porites,sp,XM35,Kiritimati_mod,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<pylipd.classes.publication.Publication object...,McGregor.2013.ChristmasIs_XM35,<pylipd.classes.location.Location object at 0x...


### PaleoData

Our next step is to create the [`PaleoData` object](https://pylipd.readthedocs.io/en/latest/api.html#pylipd.classes.paleodata.PaleoData)] and associated tables. In this case, the process is going to very simple since the coral database only contains the age and one set of measurements. 

The first thing we will need is the ability to generate unique IDs for each variable, which can be achieved with the function below:

In [32]:
import uuid

def generate_unique_id(prefix='TSC'):
    # Generate a random UUID
    random_uuid = uuid.uuid4()  # Generates a random UUID.
    
    # Convert UUID format to the specific format we need
    # UUID is usually in the form '1e2a2846-2048-480b-9ec6-674daef472bd' so we slice and insert accordingly
    id_str = str(random_uuid)
    formatted_id = f"{prefix}-{id_str[:5]}-{id_str[9:13]}-{id_str[14:18]}-{id_str[19:23]}-{id_str[24:28]}"
    
    return formatted_id

Next, let's create a list of variable names and associated usnits based on the information found in the `meas` field in the database.

In [33]:
VarName = {'\\delta^{18}O_{sw}':PaleoVariable.from_synonym('d18O'),
          '\\delta^{18}O':PaleoVariable.from_synonym('d18O'),
          'SST': PaleoVariable.from_synonym('SST')}

VarUnits = {'\\delta^{18}O_{sw}':PaleoUnit.from_synonym('per mil'),
          '\\delta^{18}O':PaleoUnit.from_synonym('per mil'),
          'SST': PaleoUnit.from_synonym('degC')}

Let's have a quick look at the name of the variable for `SST`:

In [34]:
VarName['SST'].label

'temperature'

As you can see, it is reduced to temperature, loosing some information about where the temperature is. This can be added to the interpretation object, which we will construct for each of the variables:

In [35]:
interp_d18Osw = Interpretation()
interp_d18Osw.setRank("1")
interp_d18Osw.setScope("Isotope")
interp_d18Osw.setVariable(InterpretationVariableConstants.seawaterIsotope)
interp_d18Osw.setVariableDetail("sea surface")
interp_d18Osw.setDirection("positive")

interp_d18O = [] # There are two possible interpretations for this quantity, one of which we have already established.
interp_d18O.append(interp_d18Osw)
interp1 = Interpretation()
interp1.setRank("2")
interp1.setScope("Climate")
interp1.setVariable(InterpretationVariableConstants.temperature)
interp1.setVariableDetail("sea surface")
interp1.setDirection("negative")
interp_d18O.append(interp1)

interp_sst = Interpretation()
interp_sst.setRank("1")
interp_sst.setScope("Climate")
interp_sst.setVariable(InterpretationVariableConstants.temperature)
interp_sst.setVariableDetail("sea surface")
interp_sst.setDirection("positive")


VarInterp = {'\\delta^{18}O_{sw}': [interp_d18Osw],
          '\\delta^{18}O': interp_d18O,
          'SST': [interp_sst]}

Let's also set a `Compilation` object for the records:

In [36]:
comp=Compilation()
comp.setName('HoloCoral')

Now let's create a PaleoData object for each of the row:

In [37]:
# information that will stay similar across each record:
filename = "paleo0measurement0.csv"

#List to keep our newly formed PaleoData objects:
paleoData_list = []

#Iterative Loop to create the content for each row:

for idx, row in df_cleaned.iterrows():
    paleodata = PaleoData() # Create the PaleoData object
    table = DataTable() # Create a DataTable object
    table.setFileName(filename)
    table.setMissingValue("NaN")

    # Create a Resolution object
    res = np.diff(row['chron'].flatten()).astype(float)
    Res = Resolution()
    Res.setMinValue(np.min(res))
    Res.setMaxValue(np.max(res))
    Res.setMeanValue(np.mean(res))
    Res.setMedianValue(np.median(res))

    #Creeate a PhysicalSample object
    PS = PhysicalSample()
    PS.setName(row['sample_id'].replace(" ",""))

    # Let's work with the Variables
    # Start with time
    time = Variable()
    time.setName('Year')
    time.setStandardVariable(PaleoVariableConstants.year)
    time.setColumnNumber(1)
    time.setVariableId(generate_unique_id())
    time.setUnits(PaleoUnitConstants.yr_AD)
    time.setValues(json.dumps(row['chron'].flatten().tolist()))
    # Calculate some metadata values
    time.setMinValue(np.nanmin(row['chron'].flatten().astype(float)))
    time.setMaxValue(np.nanmax(row['chron'].flatten().astype(float)))
    time.setMeanValue(np.nanmean(row['chron'].flatten().astype(float)))
    time.setMedianValue(np.nanmedian(row['chron'].flatten().astype(float)))
    # Add the resolution object
    time.setResolution(Res)
    # Add the Physical Sample object
    time.setPhysicalSamples([PS])

    #Let's do the environmental variable next
    envdata = Variable()
    envdata.setName(row['meas'])
    envdata.setStandardVariable(VarName[row['meas']])
    envdata.setColumnNumber(2)
    envdata.setVariableId(generate_unique_id())
    envdata.setUnits(VarUnits[row['meas']])
    envdata.setValues(json.dumps(row['data'].flatten().tolist()))
    envdata.setPartOfCompilation(comp) # add compilation information
    envdata.set_non_standard_property('sensorGenus',row['genus']) # add a new property for the genus
    envdata.set_non_standard_property('sensorSpecies', row['species']) # add new property for the species
    envdata.setPhysicalSamples([PS])
    envdata.setResolution(Res)
    envdata.setInterpretations(VarInterp[row['meas']])
    # Calculate some metadata values
    envdata.setMinValue(np.nanmin(row['data'].flatten().astype(float)))
    envdata.setMaxValue(np.nanmax(row['data'].flatten().astype(float)))
    envdata.setMeanValue(np.nanmean(row['data'].flatten().astype(float)))
    envdata.setMedianValue(np.nanmedian(row['data'].flatten().astype(float)))

    #Put the information back into table
    table.setVariables([time, envdata])

    # Put the table into the PaleoData object
    paleodata.setMeasurementTables([table])

    #Update the list
    paleoData_list.append(paleodata)

# Add column to the DataFrame
df_cleaned['PaleoData'] = paleoData_list

Let's have a look at our DataFrame:

In [38]:
df_cleaned.head()

,archive,lat,lon,site,reference,citekey,genus,species,sample_id,modern_id,...,enso_var_ratio,enso_var_ratio_q,enso_var_ratio_err,seasonal_amp_ratio,seasonal_amp_ratio_q,seasonal_amp_ratio_err,Publication,dataSetName,Location,PaleoData
0,coral,-20.900000,165.482000,Bayes Islet,"Correge, pers. comm., 2013",this study,Porites,sp,Bayes1,BayesM,...,NaN,NaN,NaN,NaN,NaN,NaN,<pylipd.classes.publication.Publication object...,Emile-Geay.2016.BayesIslet_Bayes1,<pylipd.classes.location.Location object at 0x...,<pylipd.classes.paleodata.PaleoData object at ...
1,coral,-20.900000,165.482000,Bayes Islet,"Correge, pers. comm., 2013",this study,Porites,sp,BayesM,BayesM,...,NaN,NaN,NaN,NaN,NaN,NaN,<pylipd.classes.publication.Publication object...,Emile-Geay.2016.BayesIslet_BayesM,<pylipd.classes.location.Location object at 0x...,<pylipd.classes.paleodata.PaleoData object at ...
2,coral,1.866667,202.583333,Christmas Is.,Woodroffe et al. [2003],Woodroffe_GRL03,Porites,sp.,XM1,Kiritimati_mod,...,NaN,NaN,NaN,NaN,NaN,NaN,<pylipd.classes.publication.Publication object...,Woodroffe.2003.ChristmasIs_XM1,<pylipd.classes.location.Location object at 0x...,<pylipd.classes.paleodata.PaleoData object at ...
3,coral,1.866667,202.583333,Christmas Is.,Woodroffe et al. [2003],Woodroffe_GRL03,Porites,sp.,XM9,Kiritimati_mod,...,NaN,NaN,NaN,NaN,NaN,NaN,<pylipd.classes.publication.Publication object...,Woodroffe.2003.ChristmasIs_XM9,<pylipd.classes.location.Location object at 0x...,<pylipd.classes.paleodata.PaleoData object at ...
4,coral,1.733333,202.800000,Christmas Is.,"McGregor et al., 2013",McGregor_ngeo2013,Porites,sp,XM35,Kiritimati_mod,...,NaN,NaN,NaN,NaN,NaN,NaN,<pylipd.classes.publication.Publication object...,McGregor.2013.ChristmasIs_XM35,<pylipd.classes.location.Location object at 0x...,<pylipd.classes.paleodata.PaleoData object at ...


Everything looks good! Time to create the final datasets!

## Creating the datasets and saving to a LiPD file

In [39]:
for idx, row in df_cleaned.iterrows():
    ds = Dataset()
    ds.setName(row['dataSetName'])
    ds.setArchiveType(ArchiveTypeConstants.Coral)
    ds.setDatasetId(generate_unique_id(prefix='HOLOCORAL'))
    ds.setPublications([row['Publication']])
    ds.setLocation(row['Location'])
    ds.setPaleoData([row['PaleoData']])

    # Put back into a LiPD object for writing
    L = LiPD()
    L.load_datasets([ds])
    L.create_lipd(ds.getName(), f"../data/lipd/{row['dataSetName']}.lpd")